In [ ]:
%run setup.py

# 02 · Preprocesamiento

Tareas: eliminación de duplicados, tratamiento de nulos, codificación de la
ubicación, creación de la variable `Antiguedad` y estandarización de predictores.
El resultado se guarda en `data/processed/` para los notebooks siguientes.

In [ ]:
# CELDA 2: Limpieza
df = load_raw()
print("Dimensiones iniciales:", df.shape)

# 1. Duplicados
duplicados = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicados eliminados: {duplicados}")

# 2. Variable objetivo: eliminar filas sin precio y precios no positivos
df = df.dropna(subset=["Price"])
df = df[df["Price"] > 0]
print("Tras eliminar precios nulos:", df.shape)

# 3. Fecha y año de venta de referencia
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
anio_venta = int(df["Date"].dt.year.max())
print(f"Año de venta de referencia: {anio_venta}")

# 4. Corrección de valores inválidos antes de imputar
invalidas_area = int((df["BuildingArea"] <= 0).sum())
df.loc[df["BuildingArea"] <= 0, "BuildingArea"] = np.nan
df.loc[(df["YearBuilt"] < 1800) | (df["YearBuilt"] > anio_venta), "YearBuilt"] = np.nan
print(f"BuildingArea <= 0 corregidas: {invalidas_area}")

# 5. Imputación de numéricas usadas (mediana, robusta ante outliers)
for columna in ["BuildingArea", "YearBuilt", "Distance"]:
    mediana = df[columna].median()
    df[columna] = df[columna].fillna(mediana)
    print(f"Nulos de {columna} imputados con la mediana ({mediana:.2f})")

# 6. Tratamiento de outliers extremos: acotar BuildingArea al percentil 99
limite_area = df["BuildingArea"].quantile(0.99)
df["BuildingArea"] = df["BuildingArea"].clip(upper=limite_area)
print(f"BuildingArea acotada al percentil 99: {limite_area:.2f} m²")

# 7. Antigüedad de la vivienda al momento de la venta
df["Antiguedad"] = (anio_venta - df["YearBuilt"]).clip(lower=0)

# 8. Codificación de la ubicación (variables dummy)
df = pd.get_dummies(df, columns=["Regionname"], drop_first=True)
print("Tras codificar Regionname:", df.shape)

## 2.1 Variables predictoras y escalado

In [ ]:
columnas_region = [c for c in df.columns if c.startswith("Regionname_")]
features = ["Rooms", "BuildingArea", "Antiguedad"] + columnas_region

X = df[features]
y = df["Price"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Predictoras utilizadas:", features)
print("Forma de X:", X.shape)
print("Nulos restantes en X:", int(X.isna().sum().sum()))

assert X.isna().sum().sum() == 0, "X no debe contener valores nulos"

In [ ]:
# 6. Guardar el dataset procesado
save_processed(df)
display(df[["Price", "Rooms", "BuildingArea", "Antiguedad"]].describe().T.round(2))